Import all necessary libraries

In [ ]:
import os
import argparse
import tiktoken
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
import time
from io import BytesIO
import openai
import asyncio
import nest_asyncio
import json
from dotenv import dotenv_values
from azure.ai.ml import MLClient
from azure.identity import EnvironmentCredential, ManagedIdentityCredential, get_bearer_token_provider
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    SearchIndex,
    SearchField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
)

Define global variables

In [ ]:
ml_client = None
workspace = None
credential = None
index_client = None
search_client = None
azure_client= None
blob_service_client = None
last_reset_time = time.time()
request_count = 0
token_count = 0
batch_count = 0
API_CONCURRENCY_LIMIT = 7
BATCH_SIZE = 10
MAX_RPM = 720  
MAX_TPM = 120000 
REQUEST_TIMEOUT = 60
lock = asyncio.Lock()
nest_asyncio.apply()
loop = asyncio.get_event_loop()

Initialization

In [ ]:
def initialize(subscription_id: str, resource_group: str, workspace_name: str, env_config_file: str, 
               client_id: str, service_endpoint: str, index_name: str, azure_endpoint: str, api_version: str):
    """
    Initializes the Azure Machine Learning workspace and client.

    Args:
        subscription_id (str): The Azure subscription ID.
        resource_group (str): The name of the resource group containing the workspace.
        workspace_name (str): The name of the Azure Machine Learning workspace.
        env_config_file (str): The path to the environment configuration file. If 'None', the variables passed in will be used.

    Returns:
        None
    """

    global ml_client, credential, blob_service_client, azure_client, index_client, search_client
    

    try:
        default_client_id = os.environ.get("DEFAULT_IDENTITY_CLIENT_ID")
        if default_client_id is not None:
            os.environ["AZURE_CLIENT_ID"] = default_client_id

        credential = ManagedIdentityCredential(client_id=client_id)

        if env_config_file != 'None':
            if os.path.isfile(env_config_file):
                config = dotenv_values(env_config_file)
                os.environ["AZURE_TENANT_ID"] = config["AZURE_TENANT_ID"]
                os.environ["AZURE_CLIENT_ID"] = config["AZURE_CLIENT_ID"]
                os.environ["AZURE_CLIENT_SECRET"] = config["AZURE_CLIENT_SECRET"]
                credential = EnvironmentCredential()

        # Initialize the blob client
        blob_service_client = BlobServiceClient(
            account_url=f"https://<storage_account_name>.blob.core.windows.net", credential=credential)

        # Initialize Azure OpenAI client
        token_provider = get_bearer_token_provider(
            credential, "https://cognitiveservices.azure.com/.default")

        azure_client = openai.AsyncAzureOpenAI(
            azure_endpoint=azure_endpoint,
            api_version=api_version,
            azure_ad_token_provider=token_provider
        )

        # Initialize Azure ML Client
        ml_client = MLClient(credential, subscription_id,
                             resource_group, workspace_name)

        # Initialize Search & Search Index Client
        index_client = SearchIndexClient(endpoint=service_endpoint, index_name=index_name, credential=credential)
        search_client = index_client.get_search_client(index_name)

        print('Initialization done.')
    except Exception as e:
        print(f"Initialization failed: {e}")
        return None

Helper function for defining the length of the embeddings

In [ ]:
async def calculate_embeddings(text: str, embedding_model: str = "text-embedding-ada-002") -> list:
    """
    Asynchronously calculates embeddings for a given text using a specified embedding model.
    Args:
        text (str): The input text for which embeddings need to be calculated.
        embedding_model (str, optional): The embedding model to use. Defaults to "text-embedding-ada-002".
    Returns:
        list: A list representing the calculated embeddings if successful.
        None: If all retry attempts fail.
    Raises:
        Exception: If an error occurs during the embedding calculation process.
    """

    for attempt in range(3):  # Retry up to 3 times
        try:
            response = await azure_client.embeddings.create(input=[text], model=embedding_model)
            return response.data[0].embedding
        except Exception as e:
            print(f"Error fetching embedding (Attempt {attempt + 1}): {e}")
            await asyncio.sleep(2 ** attempt)  # Exponential backoff
    return None  # Return None if all retries fail

Creating search index in azure search if this is the first time load

In [ ]:
def create_search_index(index_name: str, embedding_model: str):
    """
    Creates a search index in Azure Cognitive Search with specified fields and vector search configurations.
    This function defines a search index with multiple fields, including vector fields for embedding-based search.
    It calculates the embedding vector size using a sample input and configures the index accordingly. The index
    is then created using the Azure Cognitive Search client.
    Args:
        index_name (str): The name of the search index to be created.
    Returns:
        int: The size of the embedding vector used in the index, if the index is created successfully.
        None: If an error occurs during the creation of the search index.
    Raises:
        Exception: If there is an error during the creation of the search index, it is caught and logged.
    Notes:
        - The function uses an asynchronous embedding calculation function (`calculate_embeddings`) and runs it
          synchronously using `loop.run_until_complete`.
        - The index includes fields for storing metadata and vector representations of various text fields.
        - Vector search is configured using HNSW (Hierarchical Navigable Small World) algorithm.
        - Ensure that `index_client` and `embedding_model` are properly initialized before calling this function.
    """

    try:
        vector_embedding = loop.run_until_complete(calculate_embeddings("Test for embedding size", embedding_model))  # as it is a synchronous function and needs to call the async function
        
        # Identify the vector size
        embedding_vector_size = len(vector_embedding)

        # Define the index fields
        fields = [
            SimpleField(
                name="MeasureId",
                type=SearchFieldDataType.String,
                key=True,
                filterable=True, # to enable use delete_documents method
                sortable=True 
            ),
            SimpleField(
                name="LastUpdatedAt",
                type=SearchFieldDataType.String,
                filterable=True,
                sortable=True
            ),
            SearchableField(
                name="Name", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Name_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Description", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Description_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="BusinessRequirement", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="BusinessRequirement_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="InitialState", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="InitialState_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="TargetState", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="TargetState_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Analysis", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Analysis_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Remarks", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Remarks_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="Results", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="Results_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
            SearchableField(
                name="LessonsLearned", 
                type=SearchFieldDataType.String,
            ),
            SearchField(
                name="LessonsLearned_vector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                vector_search_dimensions=embedding_vector_size,
                vector_search_profile_name="my-vector-config",
            ),
        ]
        vector_search = VectorSearch(  
            profiles=[VectorSearchProfile(name="my-vector-config", algorithm_configuration_name="my-algorithms-config")],  
            algorithms=[HnswAlgorithmConfiguration(name="my-algorithms-config")],  
        )  

        # Create the search index
        search_index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search)
        result = index_client.create_index(search_index)
        
        # Print status message
        print(f"Index {result.name} created")

        return embedding_vector_size
    except Exception as e:
        print(f"Error creating search index: {e}")
        return None

Checks if data is available in the search index in Azure search

In [ ]:
def is_data_available_in_AI_search_index(index_name: str, embedding_model: str) -> pd.DataFrame:
    """
    Checks the availability of data in the Azure AI Search index and determines 
    the embedding vector size and whether an initial load is required.
    Returns:
        tuple: A tuple containing:
            - embedding_vector_size (int or None): The size of the embedding vector 
              if all fields in the index have the same dimensions, or None if no 
              dimensions are found or dimensions vary across fields.
            - initial_load (bool): A flag indicating whether the index is empty 
              (True for initial load, False otherwise).
    Raises:
        Exception: If there is an error retrieving the index configuration or 
                   checking if the index is empty.
    Notes:
        - If the index does not exist, it attempts to create the index using 
          `create_search_index`.
        - Logs details about the fields and their vector search dimensions.
        - Handles cases where no fields with vector search dimensions are found 
          or where dimensions vary across fields.
    """

    try:
        # Get the index configuration
        index = index_client.get_index(name=index_name)

        if index:
            dimensions_set = set()  # Initialize the set for dimensions
            for field in index.fields:
                if hasattr(field, 'vector_search_dimensions') and field.vector_search_dimensions:
                    print(f"Field: {field.name}, Dimensions: {field.vector_search_dimensions}")
                    dimensions_set.add(field.vector_search_dimensions)

            # Check if all dimensions are the same
            if len(dimensions_set) == 1:
                print("All fields have the same dimensions:", dimensions_set)
                embedding_vector_size = int(dimensions_set.pop())  # Safe to pop as there's only one element
            elif len(dimensions_set) == 0:
                print("No fields with 'vector_search_dimensions' found.")
                embedding_vector_size = None  # Handle no dimensions found
            else:
                print("Fields have different dimensions:", dimensions_set)
                embedding_vector_size = None  # Handle varying dimensions 
    except Exception as e:
        print(f"No index found with the name '{index_name}'. Attempting to create the index...")
        embedding_vector_size = create_search_index(index_name, embedding_model)

    # Check if index is empty
    try:
        results = search_client.search(search_text="*", top=1)
        is_empty = not any(results)   # any(results) returns value False if results is empty
    except Exception as e:
        print(f"Error checking if index is empty: {e}")
        return
    
    if is_empty: # flag used to point the folder path to be referred in blob container
        initial_load = True
    else:
        initial_load = False

    print(f"Initial load: {initial_load}")

    return embedding_vector_size, initial_load

Read the translated measures from the azure storage

In [ ]:
def read_translated_measures_from_blob_to_df(container_name: str, initial_load: bool) -> pd.DataFrame:
    """
    Reads translated measures from Azure Blob Storage and loads them into a Pandas DataFrame.
    This function retrieves Parquet files from a specified Azure Blob Storage container, processes them,
    and combines them into a single DataFrame. It handles both initial and incremental loads based on the
    `initial_load` parameter.
    Args:
        container_name (str): The name of the Azure Blob Storage container to read from.
        initial_load (bool): A flag indicating whether to perform an initial load or an incremental load.
                             If True, retrieves files with the prefix "Measures.parquet".
                             If False, retrieves files with the prefix "/translated_subset/new_or_updated_measures_subset_".
    Returns:
        pd.DataFrame: A Pandas DataFrame containing the combined and processed data from the Parquet files.
                      The DataFrame includes the following transformations:
                      - Duplicate rows are removed based on the 'measureId' column.
                      - Date columns ('lastUpdatedAt', 'createdAt', 'endDate', 'startDate') are converted to datetime.
                      - Specific columns ('measureId', 'analysis', 'results', 'lessonsLearned') are cast to string.
    Raises:
        Exception: If an error occurs during blob download or processing.
        ResourceNotFoundError: If the specified container or blobs are not found.
    Notes:
        - If no Parquet files are found, a message is printed, and the function returns None.
        - Errors during blob download or processing are logged but do not interrupt the execution for other blobs.
    """
    
    container_client = blob_service_client.get_container_client(container_name)

    try:   
        if initial_load:
            blob_prefix = "Measures.parquet"
        else:
            blob_prefix = "translated_subset/new_or_updated_measures_subset_"
        blobs = list(container_client.list_blobs(name_starts_with=blob_prefix))
        if not any(blobs):  # Check if the generator is empty
            print(f"No blobs found with prefix '{blob_prefix}' in container '{container_name}'. Exiting execution.")
            return None
    except Exception as e:
        print(f"Error accessing blobs in container '{container_name}': {e}")
        return None

    try:
        dataframes = []
        for blob in blobs:
            if blob.name.endswith('.parquet'):
                try:
                    downloaded_blob = container_client.download_blob(blob)
                    parquet_data = BytesIO(downloaded_blob.readall())
                    df = pd.read_parquet(parquet_data)
                    dataframes.append(df)
                except Exception as e:
                    print(f"Error downloading or processing blob {blob}: {e}")

        # Condition to check if the dataframes list is empty
        if dataframes:
            combined_measures_subset_df = pd.concat(dataframes, ignore_index=True)
        else:
            print("No dataframes to merge.")
            return None
        
        # Drop duplicates and data types check
        combined_measures_subset_df = combined_measures_subset_df.drop_duplicates(subset=['measureId'])

        combined_measures_subset_df['lastUpdatedAt'] = pd.to_datetime(
        combined_measures_subset_df['lastUpdatedAt'], errors='coerce')
        combined_measures_subset_df['createdAt'] = pd.to_datetime(
        combined_measures_subset_df['createdAt'])
        combined_measures_subset_df['endDate'] = pd.to_datetime(
        combined_measures_subset_df['endDate'], errors='coerce')
        combined_measures_subset_df['startDate'] = pd.to_datetime(
        combined_measures_subset_df['startDate'], errors='coerce')
        combined_measures_subset_df[['measureId','analysis','results','lessonsLearned']] = combined_measures_subset_df[['measureId','analysis','results','lessonsLearned']].astype(str)     # measureId converted to string to follow the vector table schema

        return combined_measures_subset_df

    except (Exception, ResourceNotFoundError) as e:
        print(f"An error occurred: {e}")
        return

Reduce the dataframe to include only the text columns

In [ ]:
def reduce_raw_dataframe_to_relevant_columns(df: pd.DataFrame, columns_relevant: list) -> pd.DataFrame:
    """
    Reduces a given DataFrame to only the relevant columns and formats the column names.
    Args:
        df (pd.DataFrame): The input DataFrame to be reduced.
        columns_relevant (list): A list of column names to retain in the DataFrame.
    Returns:
        pd.DataFrame: A DataFrame containing only the relevant columns with formatted column names.
        pd.Index: The updated column names of the DataFrame.
    Raises:
        Exception: If an error occurs during the reduction process, it prints an error message and returns None.
    Notes:
        - The column names in the resulting DataFrame will have their first character converted to uppercase.
        - Prints the first few rows of the reduced DataFrame for verification.
    """

    try:
        if df is None:
            print("Input DataFrame is None. Returning None.")
            return None, None
        # Reduce to relevant columns
        df = df[columns_relevant]

        # Switch to upper case only the first character
        df.columns = [column[0].upper() + column[1:] for column in df.columns]

        print("Relevant columns:")
        print(df.head())

        return df, df.columns
    except Exception as e:
        print(f"Error reducing DataFrame: {e}")
        return None, None

Initialize the embedding vectors

In [ ]:
def initialize_embedding_vectors(df_measures_to_upload: pd.DataFrame, columns_relevant: list, embedding_vector_size: int) -> pd.DataFrame:
    """
    Initializes embedding vectors for specified columns in a DataFrame.
    This function creates zero-initialized embedding vectors for the specified columns
    in the input DataFrame. It ensures that the DataFrame is modified safely by working
    on a copy and organizes the columns in a predefined order. Additionally, it converts
    the "MeasureId" column to a string type for consistency.
    Args:
        df_measures_to_upload (pd.DataFrame): The input DataFrame containing measures to upload.
        columns_relevant (list): A list of column names for which embedding vectors need to be initialized.
        embedding_vector_size (int): The size of the embedding vector to be initialized.
    Returns:
        pd.DataFrame: A DataFrame with initialized embedding vectors and organized columns.
                      Returns None if an exception occurs during processing.
    Raises:
        Exception: If any error occurs during the initialization process, it is caught and logged.
    Notes:
        - The function assumes that the input DataFrame contains specific columns such as "MeasureId",
          "LastUpdatedAt", and others listed in the column organization step.
        - The embedding vectors are initialized as zero vectors using NumPy for efficiency.
        - The function uses tqdm for progress tracking during the initialization of vectors.
    """

    try:
        if df_measures_to_upload is None:
            print("Input DataFrame is None. Returning None.")
            return None
        # Upper case for columns_relevant
        columns_relevant = [column[0].upper() + column[1:] for column in columns_relevant]

        # Initialize a zero vector using NumPy for efficiency
        zero_vector = np.zeros(embedding_vector_size).tolist()

        # Make sure you're working with a copy of the DataFrame to avoid SettingWithCopyWarning
        df_measures_to_upload = df_measures_to_upload.copy()

        for column in tqdm(columns_relevant, total=len(columns_relevant), desc="Initialize vector embeddings"):
            if column not in ["MeasureId", "LastUpdatedAt"]:
                df_measures_to_upload.loc[:, column + "_vector"] = [zero_vector.copy() for _ in range(len(df_measures_to_upload))]

        # Organize the columns
        df_measures_to_upload = df_measures_to_upload[
            [
                "MeasureId",
                "LastUpdatedAt",
                "Name",
                "Name_vector",
                "Description",
                "Description_vector",
                "BusinessRequirement",
                "BusinessRequirement_vector",
                "InitialState",
                "InitialState_vector",
                "TargetState",
                "TargetState_vector",
                "Analysis",
                "Analysis_vector",
                "Remarks",
                "Remarks_vector",
                "Results",
                "Results_vector",
                "LessonsLearned",
                "LessonsLearned_vector",
            ]
        ]

        # Convert MeasureId to string to maintain consistency with the vector table schema
        df_measures_to_upload["MeasureId"] = df_measures_to_upload["MeasureId"].astype(str)

        print("Size of data frame with measures to update: {}".format(df_measures_to_upload.shape))
        print(df_measures_to_upload.head())

        return df_measures_to_upload
    except Exception as e:
        print(f"Error initializing embedding vectors: {e}")
        return None

Helper functions to check for token size before creating the embeddings using Azure Open AI

In [ ]:
async def check_token_size_before_embedding(text: str, embedding_model: str = "text-embedding-ada-002") -> bool:
    """
    Asynchronously checks the token size of a given text before embedding.
    This function encodes the input text using the specified embedding model's tokenizer
    and calculates the number of tokens. It returns the token count as a global variable
    and indicates whether the token size was successfully calculated.
    Args:
        text (str): The input text to be tokenized and checked.
        embedding_model (str, optional): The name of the embedding model to use for tokenization.
            Defaults to "text-embedding-ada-002".
    Returns:
        bool: Always returns the token count as a global variable `n_tokens`.
    """
    
    global n_tokens
    enc = tiktoken.encoding_for_model(embedding_model)
    n_tokens = len(enc.encode(text=text))
    return n_tokens

In [ ]:
def convert_timestamp(obj: any) -> any:
    """
    Recursively converts pandas Timestamps in the given object to formatted strings.
    Args:
        obj (any): The object to convert. Can be a pandas Timestamp, a dictionary, a list, or any other type.
    Returns:
        any: The converted object with pandas Timestamps formatted as strings. If the input is a dictionary or list,
             the function will recursively convert all Timestamps within it. If the input is not a pandas Timestamp,
             dictionary, or list, it will be returned unchanged.
    """

    if isinstance(obj, pd.Timestamp):  
        return obj.strftime('%Y-%m-%d %H:%M:%S')  
    elif isinstance(obj, dict):  
        return {k: convert_timestamp(v) for k, v in obj.items()}
    elif isinstance(obj, list): 
        return [convert_timestamp(i) for i in obj]
    return obj

Helper functions to enforce tpm and rpm limits checks before sending the text for creating embeddings using Azure Open AI

In [ ]:
async def enforce_tpm_limits(tokens_used: int = 0):
    """
    Enforces a tokens-per-minute (TPM) limit by dynamically calculating and applying wait times 
    when the limit is exceeded. This function ensures that the token usage stays within the 
    specified maximum TPM.
    Args:
        tokens_used (int): The number of tokens being used in the current operation. Defaults to 0.
    Behavior:
        - Resets the token count if a new minute has started since the last reset.
        - If the token count exceeds the maximum TPM, calculates the required wait time 
          and pauses execution until tokens are available again.
        - Updates the global token count after enforcing the limits.
    Globals:
        last_reset_time (float): The timestamp of the last token count reset.
        token_count (int): The current count of tokens used within the current minute.
        lock (asyncio.Lock): A lock to ensure thread-safe updates to shared resources.
        MAX_TPM (int): The maximum number of tokens allowed per minute.
    Raises:
        None
    Example:
        await enforce_tpm_limits(tokens_used=50)
    """

    global last_reset_time, token_count
    current_time = time.time()
    async with lock:
        # Calculate when tokens will be available again
        time_since_last_reset = current_time - last_reset_time
        if time_since_last_reset >= 60:  # Reset if a new minute has started
            token_count = 0
            last_reset_time = current_time
        
        # If exceeding TPM, calculate the wait time dynamically
        if token_count + tokens_used > MAX_TPM:
            wait_time = 60 - time_since_last_reset
            print(f"TPM limit reached. Waiting for {wait_time:.2f} seconds...")
            await asyncio.sleep(wait_time)
            token_count = 0  # Reset after sleep
        
        # Update token count after enforcing limits
        token_count += tokens_used
        print(f"Token count: {token_count}")


async def enforce_rpm_limits():
    """
    Asynchronous function to enforce rate limits based on requests per minute (RPM).
    This function ensures that the number of requests made does not exceed the 
    configured maximum RPM (`MAX_RPM`). If the limit is reached, it calculates the 
    necessary wait time until the next minute and pauses execution accordingly.
    Global Variables:
        last_reset_time (float): The timestamp of the last reset of the request count.
        request_count (int): The current count of requests made within the current minute.
    Behavior:
        - Resets the request count if a new minute starts.
        - If the request count exceeds `MAX_RPM`, calculates the remaining time in the 
          current minute and pauses execution for that duration.
        - Increments the request count after ensuring compliance with the RPM limit.
    Usage:
        This function should be called before making a request to ensure that the 
        rate limits are not exceeded.
    Note:
        This function uses an asynchronous lock (`lock`) to ensure thread-safe 
        updates to shared variables in concurrent environments.
    """

    global last_reset_time, request_count
    current_time = time.time()
    async with lock:
        # Reset if a new minute starts
        time_since_last_reset = current_time - last_reset_time
        if time_since_last_reset >= 60:
            request_count = 0
            last_reset_time = current_time

        # If exceeding RPM, dynamically calculate wait time
        if request_count >= MAX_RPM:
            wait_time = 60 - time_since_last_reset  
            print(f"RPM limit reached. Waiting for {wait_time:.2f} seconds...")
            await asyncio.sleep(wait_time)
            request_count = 0  # Reset after sleep

        # Update request count
        request_count += 1
        print(f"Request count: {request_count}")

Upload to azure search after embeddings are created

In [ ]:
async def upload_to_azure_search(rows_to_upload: list, batch_size: int=10):
    """
    Asynchronously uploads data to Azure AI Search in batches with concurrency control.
    Args:
        rows_to_upload (list): A list of rows to be uploaded to Azure AI Search.
        batch_size (int, optional): The number of rows to include in each batch. Defaults to 10.
    Returns:
        list: A list of measure IDs that were successfully uploaded to Azure AI Search.
    Notes:
        - The function uses a semaphore to limit the number of concurrent uploads to 10.
        - Rows are split into batches of size `batch_size` for uploading.
        - Any errors during task queuing are logged, and empty batches are skipped.
        - The uploaded measure IDs are collected from the result queue and returned.
    """

    tasks = []
    semaphore = asyncio.Semaphore(10)  # Limit to 5 concurrent uploads
    result_queue = asyncio.Queue() # to capture the measureIds uploaded in Azure AI Search

    # Split rows_to_upload into chunks for batch uploading
    for start in range(0, len(rows_to_upload), batch_size):
        end = start + batch_size
        chunk = rows_to_upload[start:end]
        
        # Ensure chunk is not empty
        if not chunk:
            print(f"Empty batch, skipping upload.")
            continue

        try:
            # Use the semaphore to limit concurrent uploads
            tasks.append(upload_chunk_with_semaphore(chunk, semaphore, result_queue))
            
        except Exception as e:
            print(f"Error queuing batch {start} to {end}: {str(e)}")

    # Wait for all tasks to finish
    await asyncio.gather(*tasks)

    # Collect all measure_ids from the result queue
    new_measure_ids = []
    while not result_queue.empty():
        new_measure_ids.extend(await result_queue.get())

    return new_measure_ids

async def upload_chunk_with_semaphore(chunk_to_upload: list, semaphore: int, result_queue: str):
    """
    Asynchronously uploads a chunk of documents to Azure Search while managing concurrency using a semaphore.
    Args:
        chunk_to_upload (list): A list of documents to be uploaded. Each document is expected to be a dictionary.
        semaphore (asyncio.Semaphore): A semaphore object to limit the number of concurrent uploads.
        result_queue (asyncio.Queue): An asyncio queue to store the MeasureIds of successfully uploaded documents.
    Returns:
        None
    Raises:
        Exception: If an error occurs during the upload process, it will be caught and logged.
    Notes:
        - The function uses `asyncio.to_thread` to execute the `upload_documents` method in a thread-safe manner.
        - MeasureIds from the uploaded documents are extracted and logged for tracking purposes.
        - Successfully uploaded MeasureIds are added to the result queue for further processing.
    """

    async with semaphore:
        try:
            # Extract MeasureId for logging
            measure_ids = [row.get("MeasureId") for row in chunk_to_upload]
            print(f"Uploading a chunk of documents for MeasureIds: {measure_ids}")

            # Upload the chunk to Azure Search
            await asyncio.to_thread(search_client.upload_documents, documents=chunk_to_upload)
            print(f"Upload completed for the chunk of MeasureIds: {measure_ids}")

            # Safely add measure_ids to the result queue
            await result_queue.put(measure_ids)

        except Exception as e:
            print(f"Error uploading chunk: {e}")

Define the batches to be created for creating the embeddings using Azure Open AI. Here concurrency has been implemented to speed up the processing time.

In [ ]:
async def log_errors(errors: list,log_container_name: str):
    """
    Logs a list of errors to an Azure Blob Storage container asynchronously.
    This function converts the provided list of errors into a JSON format and uploads it
    to a specified Azure Blob Storage container. If the container does not exist, it will
    be created. The upload process is performed using a separate thread to avoid blocking
    the event loop.
    Args:
        errors (list): A list of error messages or objects to be logged.
        log_container_name (str): The name of the Azure Blob Storage container where the
                                error logs will be stored.
    Raises:
        Exception: If an error occurs during the upload process, it will be caught and
                printed to the console.
    Notes:
        - The error logs are stored in the container under the "errors/" prefix with a
        timestamped filename in the format `errors_YYYYMMDD_HHMMSS.json`.
        - The function ensures that all upload tasks are completed before exiting.
    Example:
        errors = [{"error": "File not found", "timestamp": "2023-01-01T12:00:00Z"}]
        await log_errors(errors, "my-log-container")
    """

    container_client = blob_service_client.get_container_client(log_container_name)
    if not container_client.exists():
        print("Log container does not exist. Creating container..")
        blob_service_client.create_container(log_container_name)

    # Convert errors to JSON and upload to Blob Storage
    json_errors = json.dumps(errors)
    blob_prefix = 'errors/'
    blob_client = container_client.get_blob_client(f"{blob_prefix}errors_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    print("Errors have been prepared for logging.")
    
    # Function to upload the error data to the blob
    async def upload_error():
        try:
            # Upload the errors to the blob using a separate thread to avoid blocking the event loop
            await asyncio.to_thread(blob_client.upload_blob, json_errors, overwrite=False)
            print(f"Successfully uploaded error logs to {blob_client.blob_name}")
        except Exception as e:
            print(f"Error uploading error log: {e}")

    # List of tasks to track all uploads (if there are multiple blobs or uploads)
    tasks = []
    tasks.append(upload_error())

    # Wait for all tasks to complete before exiting
    await asyncio.gather(*tasks)
    
    print("All error logging tasks have been processed and completed.")

    

async def process_batch_async(batch: pd.DataFrame, columns_embeddings: list, semaphore: int, 
                              processed_docs: set, embedding_vector_size: int,new_measure_ids: list, embedding_model: str, max_token_for_model: int):
    """
    Asynchronously processes a batch of data, generates embeddings for specified columns, 
    and uploads the processed data to Azure Search with concurrency control.
    Args:
        batch (pd.DataFrame): The batch of data to process, represented as a pandas DataFrame.
        columns_embeddings (list): List of column names for which embeddings need to be generated.
        semaphore (int): Semaphore to control concurrency for asynchronous operations.
        processed_docs (set): A set of MeasureIds that have already been processed to avoid duplication.
        embedding_vector_size (int): Maximum allowed size for the embedding vector.
        new_measure_ids (list): A list to store MeasureIds of newly processed rows.
    Returns:
        tuple: A tuple containing:
            - new_measure_ids (list): Updated list of MeasureIds for newly processed rows.
            - rows_with_errors (list or None): List of rows with errors, including details of the error and column, 
              or None if no errors occurred.
    Raises:
        Exception: If any error occurs during embedding generation or data upload.
    Notes:
        - The function skips rows that have already been processed based on their MeasureId.
        - It checks token limits (both per request and per minute) before generating embeddings.
        - Rows with token counts exceeding the embedding vector size are skipped.
        - Errors during embedding generation or upload are logged and returned for further inspection.
    """

    async with semaphore:
        # Get all MeasureIds for the current batch
        batch_ids = [row["MeasureId"] for row in batch]

        # processed_docs is a set to store the document IDs that have been processed. Re-initializes to set() for every new run.
        unprocessed_rows = [row for row in batch if row["MeasureId"] not in processed_docs]
        if not unprocessed_rows:
            print(f"Skipping batch, all rows already processed: {batch_ids}")
            return None
        
        # Prepare list for rows to upload to blob storage
        rows_to_upload = []

        # Iterate over each row and each column, sending individual values for embedding
        rows_with_errors = []
        token_count_exceed_rows = []
        for row in unprocessed_rows:
            row_data = {"MeasureId": row["MeasureId"], "LastUpdatedAt": row["LastUpdatedAt"]}

            for col in columns_embeddings:
                value = row.get(col)
                # Only process non-null, non-None, and non-blank values
                if value is not None and value.strip() != "" and value != 'None':  # Check if the value is not null or blank
                    try:
                        # Check the token size before processing the embedding
                        token_count = await check_token_size_before_embedding(value, embedding_model)
                        if token_count > max_token_for_model:
                            print(f"Token count ({token_count}) exceeds limit. Skipping: {value}")
                            token_count_exceed_rows.append({"row": row, "column": col})
                            continue

                        # Check the RPM limit (requests per minute) before sending a request for the embedding
                        await enforce_rpm_limits()
                        # Check the TPM limit (tokens per minute) before sending a request for the embedding
                        await enforce_tpm_limits(tokens_used=token_count)

                        # Call the embeddings creation function for each valid value immediately
                        embedding = await calculate_embeddings(value, embedding_model)

                        # Map the embedding and the original text back to the original row
                        row[f"{col}_vector"] = embedding  # Store embedding in the row
                        row_data[col] = row.get(col)  # Store the original text in the row
                        row_data[f"{col}_vector"] = embedding  # Store the embedding in the row_data


                    except Exception as e:
                        rows_with_errors.append({"row": row, "error": str(e), "column": col})

                else:
                    # If the value is None or blank, set the embedding to None
                    row_data[col] = row.get(col)
                    row_data[f"{col}_vector"] = row.get(f"{col}_vector")


            # Add the row to the embeddings_data list
            rows_to_upload.append(row_data)


        try:
            # Instead of overwriting, we append the new_measure_ids to the passed list
            new_measure_ids_batch = await upload_to_azure_search(rows_to_upload)
            if new_measure_ids_batch:
                new_measure_ids.extend(new_measure_ids_batch)  # Append the result to the existing list
        except Exception as e:
            print(f"Failed to upload embeddings for batch {batch_ids}: {str(e)}")

    # Return the new_measure_ids and any rows with errors
    if len(rows_with_errors) > 0:
        print(f"Rows with errors: {rows_with_errors}")
        return new_measure_ids, rows_with_errors
    else:
        return new_measure_ids, None
    

async def create_batches_for_embeddings_creation(df: pd.DataFrame, columns_embeddings: list, semaphore: int, processed_docs: set, 
                                                 log_container_name: str, embedding_vector_size: int, embedding_model: str, max_token_for_model: int):
    """
    Creates batches from a DataFrame and processes them asynchronously to generate embeddings.
    This function divides the input DataFrame into batches, processes each batch concurrently,
    and collects the results. It also handles errors encountered during processing and logs them.
    Args:
        df (pd.DataFrame): The input DataFrame containing the data to be processed.
        columns_embeddings (list): A list of column names to be used for generating embeddings.
        semaphore (int): The maximum number of concurrent tasks allowed.
        processed_docs (set): A set of already processed document identifiers to avoid duplication.
        log_container_name (str): The name of the log container for storing error logs.
        embedding_vector_size (int): The size of the embedding vectors to be generated.
    Returns:
        list: A list of new measure IDs generated during the processing.
    Raises:
        Exception: If any unexpected error occurs during batch processing.
    Notes:
        - The function uses asyncio.gather to run tasks concurrently.
        - Errors encountered during processing are logged using the `log_errors` function.
        - The `new_measure_ids` list is updated in-place during batch processing.
    """

    global results
    tasks = []  # List of tasks to run concurrently
    errors = []  # List to store rows with errors
    new_measure_ids = []  # Initialize the list to accumulate measure ids
    
    if df is None:
        print("Input DataFrame is None. Returning None.")
        return None
    for start in range(0, len(df), BATCH_SIZE):
        batch = df.iloc[start:start + BATCH_SIZE].to_dict(orient="records")
        print(f"Queueing batch {start} to {start + BATCH_SIZE}")

        # Pass new_measure_ids to the process_batch_async so it gets updated
        tasks.append(process_batch_async(batch, columns_embeddings, semaphore, processed_docs, 
                                         embedding_vector_size, new_measure_ids, embedding_model, max_token_for_model))

    # Run all the tasks concurrently and await their completion
    results = await asyncio.gather(*tasks)

    # Since the new_measure_ids was passed and updated inside each batch, it already contains all the measure ids.
    # No need to extract anything from `results` here, as the list is already updated.

    # Gather all rows with errors from the result of the tasks
    for result in results:
        if result and isinstance(result, tuple):
            rows_with_errors = result[1]
            if rows_with_errors:
                errors.extend(rows_with_errors)

    # Handle the rows with errors (logging)
    if errors:
        print(f"Found {len(errors)} rows with errors. Logging errors..")
        await log_errors(errors, log_container_name)
    else:
        print("No errors found.")

    return new_measure_ids

Logging

In [ ]:
def log_uploaded_measureids(new_measure_ids: list, log_container_name: str):
    """
    Logs a list of measure IDs to an Azure Blob Storage container. If the specified container does not exist, 
    it will be created. The measure IDs are stored in a text file named with a timestamp.
    Args:
        new_measure_ids (list): A list of measure IDs to be logged. If the list is empty, the function will exit early.
        log_container_name (str): The name of the Azure Blob Storage container where the measure IDs will be logged.
    Returns:
        None
    Behavior:
        - If the `new_measure_ids` list is empty, the function prints a message and exits.
        - Creates a new text file with the current timestamp in the `uploaded_measureIds` directory.
        - If the specified container does not exist, it creates the container.
        - If a blob with the same name already exists, it appends the new measure IDs to the existing content.
        - Uploads the updated content to the Azure Blob Storage container.
    Exceptions:
        - Prints an error message if uploading the measure IDs fails.
    """

    if not new_measure_ids:  # Check if the list is empty
        print("No measureIds to log.")
        return
    
    measure_ids_filename = f"uploaded_measureIds/measureIds_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    measure_ids_content = "\n".join(map(str, new_measure_ids))


    measure_ids_blob_client = blob_service_client.get_blob_client(container=log_container_name, blob=measure_ids_filename)
    
    container_client = blob_service_client.get_container_client(log_container_name)
    if not container_client.exists():
        print(f"Log container '{log_container_name}' does not exist. Creating container...")
        blob_service_client.create_container(log_container_name)

    try:
        # Try to download the current content of the blob if it exists
        existing_content = ""
        try:
            existing_content = measure_ids_blob_client.download_blob().readall().decode("utf-8")
        except Exception:
            pass

        # Append the new measureIds to the existing content
        updated_content = existing_content + "\n" + measure_ids_content if existing_content else measure_ids_content
        measure_ids_blob_client.upload_blob(updated_content, overwrite=True)
        print(f"Uploaded updated measureIds to {measure_ids_filename}")
    
    except Exception as e:
        print(f"Failed to upload measureIds to {measure_ids_filename}: {e}")

Update the azure container with the deleted measures subset

In [ ]:
def get_deleted_measures(container_name: str):
    """
    Retrieves and processes deleted measures stored as Parquet files in an Azure Blob Storage container.
    This function lists all blobs in the specified container that match a predefined prefix and have a `.parquet` 
    extension. It downloads these blobs, reads them into Pandas DataFrames, and combines them into a single DataFrame.
    Args:
        container_name (str): The name of the Azure Blob Storage container to retrieve the deleted measures from.
    Returns:
        pd.DataFrame or None: A combined Pandas DataFrame containing the data from all the Parquet files, or None 
        if no dataframes are found or an error occurs.
    Raises:
        Exception: If an error occurs during blob download or processing, it is caught and logged, but the function 
        continues processing other blobs.
        ResourceNotFoundError: If the specified container or blobs are not found, it is caught and logged.
    """
    try:
        container_client = blob_service_client.get_container_client(container_name)
        blob_prefix = "translated_subset/deleted_measures_subset_"
        blobs = list(container_client.list_blobs(name_starts_with=blob_prefix))
        if not blobs:  # Check if the list of blobs is empty
            print(f"No blobs found with prefix '{blob_prefix}' in container '{container_name}'. Exiting function.")
            return None
    except ResourceNotFoundError as e:
        print(f"Error: The specified container '{container_name}' or blobs with prefix '{blob_prefix}' were not found. {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while accessing blobs: {e}")
        return None

    try: 
        # Loop through each parquet blob, download and process --> not needed because we are combining dataframes in initial load
        dataframes = []
        for blob in blobs:
            print(blob.name)
            if blob.name.endswith('.parquet'):
                try:
                    downloaded_blob = container_client.download_blob(blob)
                    parquet_data = BytesIO(downloaded_blob.readall())
                    df = pd.read_parquet(parquet_data)
                    dataframes.append(df)
                except Exception as e:
                    print(f"Error downloading or processing blob {blob.name}: {e}")

        # Condition to check if the dataframes list is empty
        if not dataframes:
            print("No dataframes to merge. Exiting function. ")
            return None

        combined_measures_subset_df = pd.concat(dataframes, ignore_index=True)

        return combined_measures_subset_df

    except (Exception, ResourceNotFoundError) as e:
        print(f"An error occurred: {e}")
        return
    

def delete_measures_from_azure_search(container_name: str):
    """
    Deletes measures from Azure Search based on the provided container name.
    This function retrieves a list of measures marked for deletion from the specified
    container, and then deletes them from Azure Search using the search client.
    Args:
        container_name (str): The name of the container from which to retrieve measures
                              marked for deletion.
    Returns:
        None: Prints a message indicating whether there were measures to delete and
              whether the deletion succeeded.
    Notes:
        - The function assumes the existence of a `get_deleted_measures` function that
          retrieves measures marked for deletion from the specified container.
        - The `search_client` object must be properly initialized and configured to
          interact with Azure Search.
        - The `measures_to_delete` DataFrame is expected to have a column named
          'measureId' containing the IDs of the measures to delete.
    """

    measures_to_delete = get_deleted_measures(container_name)
    if measures_to_delete is None or measures_to_delete.empty:
        print("No measures to delete.")
        return
    measures_ids_to_delete = measures_to_delete['measureId'].astype(str).tolist()

    result = search_client.delete_documents(documents=[{"MeasureId": measure_id} for measure_id in measures_ids_to_delete])

    print("Deletion of measure Ids succeeded: {}".format(result[0].succeeded))